In [3]:
%run nb_dq_utils

StatementMeta(, 20f59592-565c-49db-bfbb-9fb03f52a7ce, 16, Finished, Available, Finished, True)

In [3]:
from pyspark.sql import functions as F
logger_fact = get_logger("gold.fact_sales.dq")

# ── Read Silver ───────────────────────────────────────────────────────
df_sales_orders = spark.read.table("lh_adventureworks_silver.sales.sales_orders")

logger_fact.info(f"Silver sales_orders rows: {df_sales_orders.count()}")
#logger.info(f"{df_sales_orders.printSchema()}")

StatementMeta(, 325cde5c-f8a8-4d43-a7eb-2373c46f124f, 11, Finished, Available, Finished, False)

INFO:gold.fact_sales.dq:Silver sales_orders rows: 121317


In [4]:
# ── Generate dim_date ───────────────────────────────────────────────────
from pyspark.sql import functions as F

date_range = df_sales_orders.select(
    F.min("OrderDate").alias("min_date"),
    F.max("OrderDate").alias("max_date")
).collect()[0]

logger_fact.info(f"Date range: {date_range['min_date']} to {date_range['max_date']}")

df_dim_date = spark.sql(f"""
    SELECT
        explode(sequence(
            to_date('{date_range['min_date']}'),
            to_date('{date_range['max_date']}'),
            interval 1 day
        )) AS FullDate
""")

df_dim_date = df_dim_date.select(
    F.date_format("FullDate", "yyyyMMdd").cast("integer").alias("DateKey"),
    F.col("FullDate"),
    F.year("FullDate").alias("Year"),
    F.quarter("FullDate").alias("Quarter"),
    F.month("FullDate").alias("Month"),
    F.date_format("FullDate", "MMMM").alias("MonthName"),
    F.weekofyear("FullDate").alias("WeekOfYear"),
    F.dayofmonth("FullDate").alias("DayOfMonth"),
    F.dayofweek("FullDate").alias("DayOfWeek"),
    F.date_format("FullDate", "EEEE").alias("DayName"),
    F.when(F.dayofweek("FullDate").isin([1, 7]), True).otherwise(False).alias("IsWeekend")
)

logger_fact.info(f"dim_date rows: {df_dim_date.count()}")

StatementMeta(, 325cde5c-f8a8-4d43-a7eb-2373c46f124f, 12, Finished, Available, Finished, False)

INFO:gold.fact_sales.dq:Date range: 2022-05-30 00:00:00 to 2025-06-29 00:00:00
INFO:gold.fact_sales.dq:dim_date rows: 1127


In [5]:
# ── Write dim_date ────────────────────────────────────────────────────
#spark.sql("CREATE SCHEMA IF NOT EXISTS lh_adventureworks_gold.sales")

df_dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("lh_adventureworks_gold.sales.dim_date")

logger_fact.info("gold.sales.dim_date written successfully.")

StatementMeta(, 325cde5c-f8a8-4d43-a7eb-2373c46f124f, 13, Finished, Available, Finished, False)

INFO:gold.fact_sales.dq:gold.sales.dim_date written successfully.


In [6]:
# ── Build fact_sales ────────────────────────────────────────────────────
df_fact_sales = df_sales_orders.select(
    "SalesOrderID",
    "SalesOrderDetailID",
    F.date_format("OrderDate", "yyyyMMdd").cast("integer").alias("OrderDateKey"),
    F.date_format("DueDate", "yyyyMMdd").cast("integer").alias("DueDateKey"),
    F.date_format("ShipDate", "yyyyMMdd").cast("integer").alias("ShipDateKey"),
    "CustomerID",
    "SalesPersonID",
    "TerritoryID",
    "ProductID",
    "SpecialOfferID",
    "BillToAddressID",
    "ShipToAddressID",
    "Status",
    "OnlineOrderFlag",
    "OrderQty",
    "UnitPrice",
    "UnitPriceDiscount",
    "LineTotal",
    "SubTotal",
    "TaxAmt",
    "Freight",
    "TotalDue"
)

actual_rows = df_fact_sales.count()
logger_fact.info(f"fact_sales row count: {actual_rows}")
#logger.info(f"{df_fact_sales.printSchema()}")

StatementMeta(, 325cde5c-f8a8-4d43-a7eb-2373c46f124f, 14, Finished, Available, Finished, False)

INFO:gold.fact_sales.dq:fact_sales row count: 121317


In [9]:
# These weren't read yet in this notebook — needed for the referential integrity checks
df_product_silver = spark.read.table("lh_adventureworks_silver.production.product")
df_territory_silver = spark.read.table("lh_adventureworks_silver.sales.territory")
df_dim_date = spark.read.table("lh_adventureworks_gold.sales.dim_date")
checks = [
    check_row_count(df_fact_sales, df_sales_orders.count()),
    check_duplicate_pk(df_fact_sales, ["SalesOrderID", "SalesOrderDetailID"]),
    check_null_pk(df_fact_sales, "OrderDateKey"),
    check_null_pk(df_fact_sales, "ProductID"),
    check_null_pk(df_fact_sales, "CustomerID"),
    check_null_pk(df_fact_sales, "TerritoryID"),
    check_referential_integrity(df_fact_sales, df_dim_date, "OrderDateKey", dim_join_col="DateKey", dim_name="dim_date"),
    check_referential_integrity(df_fact_sales, df_product_silver, "ProductID", dim_name="product"),
    check_referential_integrity(df_fact_sales, df_territory_silver, "TerritoryID", dim_name="territory"),
]

run_dq_checks(checks, logger_fact, "fact_sales")

StatementMeta(, 325cde5c-f8a8-4d43-a7eb-2373c46f124f, 17, Finished, Available, Finished, False)

INFO:gold.fact_sales.dq:[DQ] [PASS] Row count — Expected 121,317, got 121,317
INFO:gold.fact_sales.dq:[DQ] [PASS] Duplicate PK — SalesOrderID, SalesOrderDetailID — 0 duplicates
INFO:gold.fact_sales.dq:[DQ] [PASS] Null PK — OrderDateKey — 0 nulls
INFO:gold.fact_sales.dq:[DQ] [PASS] Null PK — ProductID — 0 nulls
INFO:gold.fact_sales.dq:[DQ] [PASS] Null PK — CustomerID — 0 nulls
INFO:gold.fact_sales.dq:[DQ] [PASS] Null PK — TerritoryID — 0 nulls
INFO:gold.fact_sales.dq:[DQ] [PASS] Referential integrity — OrderDateKey → dim_date — 0 rows with OrderDateKey not found in dim_date
INFO:gold.fact_sales.dq:[DQ] [PASS] Referential integrity — ProductID → product — 0 rows with ProductID not found in product
INFO:gold.fact_sales.dq:[DQ] [PASS] Referential integrity — TerritoryID → territory — 0 rows with TerritoryID not found in territory
INFO:gold.fact_sales.dq:[DQ] All checks passed for fact_sales.


In [11]:
# ── Write ──────────────────────────────────────────────────────────────
df_fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("lh_adventureworks_gold.sales.fact_sales")

df_verify = spark.read.table("lh_adventureworks_gold.sales.fact_sales")
logger_fact.info(f"gold.sales.fact_sales written — {df_verify.count():,} rows verified.")

StatementMeta(, 325cde5c-f8a8-4d43-a7eb-2373c46f124f, 19, Finished, Available, Finished, False)

INFO:gold.fact_sales.dq:gold.sales.fact_sales written — 121,317 rows verified.


In [12]:
logger_prod = get_logger("gold.dim_product.dq")
# ── Read Silver product ──────────────────────────────────────────────
df_product_silver = spark.read.table("lh_adventureworks_silver.production.product")

logger_prod.info(f"Silver product rows: {df_product_silver.count()}")

# ── Build dim_product ───────────────────────────────────────────────
df_dim_product = df_product_silver.select(
    "ProductID",
    "Name",
    "ProductNumber",
    "Color",
    "Size",
    "Weight",
    "StandardCost",
    "ListPrice",
    "ProductSubcategoryID",
    "SubcategoryName",
    "ProductCategoryID",
    "CategoryName",
    "ProductLine",
    "Class",
    "Style",
    "SellStartDate",
    "SellEndDate",
    "DiscontinuedDate",
    F.when(F.col("DiscontinuedDate").isNull(), True).otherwise(False).alias("IsActive")
)

actual_rows = df_dim_product.count()
logger_prod.info(f"dim_product row count: {actual_rows}")


StatementMeta(, 325cde5c-f8a8-4d43-a7eb-2373c46f124f, 20, Finished, Available, Finished, False)

INFO:gold.dim_product.dq:Silver product rows: 504
INFO:gold.dim_product.dq:dim_product row count: 504


In [14]:
checks = [
    check_row_count(df_dim_product, df_product_silver.count()),
    check_duplicate_pk(df_dim_product, "ProductID"),
    check_null_pk(df_dim_product, "ProductID"),
]
run_dq_checks(checks, logger_prod, "dim_product")

StatementMeta(, 325cde5c-f8a8-4d43-a7eb-2373c46f124f, 22, Finished, Available, Finished, False)

INFO:gold.dim_product.dq:[DQ] [PASS] Row count — Expected 504, got 504
INFO:gold.dim_product.dq:[DQ] [PASS] Duplicate PK — ProductID — 0 duplicates
INFO:gold.dim_product.dq:[DQ] [PASS] Null PK — ProductID — 0 nulls
INFO:gold.dim_product.dq:[DQ] All checks passed for dim_product.


In [16]:
#── Write ──────────────────────────────────────────────────────────────
spark.sql("CREATE SCHEMA IF NOT EXISTS lh_adventureworks_gold.production")

df_dim_product.write \
    .format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("lh_adventureworks_gold.production.dim_product")

logger_prod.info(f"gold.production.dim_product written — {spark.read.table('lh_adventureworks_gold.production.dim_product').count():,} rows verified.")

StatementMeta(, 325cde5c-f8a8-4d43-a7eb-2373c46f124f, 24, Finished, Available, Finished, False)

INFO:gold.dim_product.dq:gold.production.dim_product written — 504 rows verified.


In [20]:
# ── Read Silver territory ────────────────────────────────────────────
df_territory_silver = spark.read.table("lh_adventureworks_silver.sales.territory")

logger_ter = get_logger("gold.dim_territory.dq")
logger_ter.info(f"Silver territory rows: {df_territory_silver.count()}")

StatementMeta(, 325cde5c-f8a8-4d43-a7eb-2373c46f124f, 28, Finished, Available, Finished, False)

INFO:gold.dim_territory.dq:Silver territory rows: 10


In [21]:

# ── Build dim_territory ─────────────────────────────────────────────
df_dim_territory = df_territory_silver.select(
    "TerritoryID",
    "TerritoryName",
    "CountryRegionCode",
    "TerritoryGroup",
    "SalesYTD",
    "SalesLastYear",
    "CostYTD",
    "CostLastYear"
)

actual_rows = df_dim_territory.count()
logger_ter.info(f"dim_territory row count: {actual_rows}")

StatementMeta(, 325cde5c-f8a8-4d43-a7eb-2373c46f124f, 29, Finished, Available, Finished, False)

INFO:gold.dim_territory.dq:dim_territory row count: 10


In [23]:
checks = [
    check_row_count(df_dim_territory, df_territory_silver.count()),
    check_null_pk(df_dim_territory, "TerritoryID"),
    check_duplicate_pk(df_dim_territory, "TerritoryID"),
]
run_dq_checks(checks, logger_ter, "dim_territory")

StatementMeta(, 325cde5c-f8a8-4d43-a7eb-2373c46f124f, 31, Finished, Available, Finished, False)

INFO:gold.dim_territory.dq:[DQ] [PASS] Row count — Expected 10, got 10
INFO:gold.dim_territory.dq:[DQ] [PASS] Null PK — TerritoryID — 0 nulls
INFO:gold.dim_territory.dq:[DQ] [PASS] Duplicate PK — TerritoryID — 0 duplicates
INFO:gold.dim_territory.dq:[DQ] All checks passed for dim_territory.


In [24]:
df_dim_territory.write \
    .format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("lh_adventureworks_gold.sales.dim_territory")

logger_ter.info(f"gold.sales.dim_territory written — {spark.read.table('lh_adventureworks_gold.sales.dim_territory').count():,} rows verified.")

StatementMeta(, 325cde5c-f8a8-4d43-a7eb-2373c46f124f, 32, Finished, Available, Finished, False)

INFO:gold.dim_territory.dq:gold.sales.dim_territory written — 10 rows verified.


In [ ]:
# ── DQ checks ─────────────────────────────────────────────────────────
dq_results = []
def log_check(name, passed, message):
    status = "PASS" if passed else "FAIL"
    (logger.info if passed else logger.error)(f"[DQ] [{status}] {name} — {message}")
    dq_results.append({"check": name, "status": status})

log_check("Row count", actual_rows == df_territory_silver.count(), f"{actual_rows} rows")
dup_pk = actual_rows - df_dim_territory.select("TerritoryID").distinct().count()
log_check("Duplicate PK", dup_pk == 0, f"{dup_pk} duplicates")

failed = [r for r in dq_results if r["status"] == "FAIL"]
if failed:
    raise Exception(f"dim_territory DQ failed: {', '.join([r['check'] for r in failed])}")

logger.info("[DQ] All checks passed.")

# ── Write ──────────────────────────────────────────────────────────────
df_dim_territory.write \
    .format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("lh_adventureworks_gold.sales.dim_territory")

logger.info(f"gold.sales.dim_territory written — {spark.read.table('lh_adventureworks_gold.sales.dim_territory').count():,} rows verified.")

In [23]:
# ── Read Silver employee ─────────────────────────────────────────────
df_employee_silver = spark.read.table("lh_adventureworks_silver.hr.employee")

logger_emp = get_logger("gold.dim_employee.dq")
logger_emp.info(f"Silver employee rows: {df_employee_silver.count()}")

StatementMeta(, db04bf04-00d2-4222-8cf9-9501d4709410, 30, Finished, Available, Finished, False)

INFO:gold.dim_employee.dq:Silver employee rows: 290


In [24]:
# ── Build dim_employee ──────────────────────────────────────────────
df_dim_employee = df_employee_silver.select(
    "EmployeeID",
    "EmployeeFullName",
    "FirstName",
    "LastName",
    "JobTitle",
    "Gender",
    "MaritalStatus",
    "BirthDate",
    "HireDate",
    "SalariedFlag",
    "CurrentFlag",
    "DepartmentID",
    "DepartmentName",
    "GroupName",
    "OrganizationLevel",
    "OrganizationNode"
)

actual_rows = df_dim_employee.count()
logger_emp.info(f"dim_employee row count: {actual_rows}")

StatementMeta(, db04bf04-00d2-4222-8cf9-9501d4709410, 31, Finished, Available, Finished, False)

INFO:gold.dim_employee.dq:dim_employee row count: 290


In [25]:
# ── DQ checks ─────────────────────────────────────────────────────────
checks = [
    check_row_count(df_dim_employee, df_employee_silver.count()),
    check_null_pk(df_dim_employee, "EmployeeID"),
    check_duplicate_pk(df_dim_employee, "EmployeeID"),
    check_join_coverage(df_dim_employee, "DepartmentID", "DepartmentName"),
]
run_dq_checks(checks, logger_emp, "dim_employee")

StatementMeta(, db04bf04-00d2-4222-8cf9-9501d4709410, 32, Finished, Available, Finished, False)

INFO:gold.dim_employee.dq:[DQ] [PASS] Row count — Expected 290, got 290
INFO:gold.dim_employee.dq:[DQ] [PASS] Null PK — EmployeeID — 0 nulls
INFO:gold.dim_employee.dq:[DQ] [PASS] Duplicate PK — EmployeeID — 0 duplicates
INFO:gold.dim_employee.dq:[DQ] [PASS] DepartmentName join coverage — 0 rows with DepartmentID but no DepartmentName
INFO:gold.dim_employee.dq:[DQ] All checks passed for dim_employee.


In [26]:
# ── Write ──────────────────────────────────────────────────────────────
#spark.sql("CREATE SCHEMA IF NOT EXISTS lh_adventureworks_gold.hr")

df_dim_employee.write \
    .format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("lh_adventureworks_gold.hr.dim_employee")

logger_emp.info(f"gold.hr.dim_employee written — {spark.read.table('lh_adventureworks_gold.hr.dim_employee').count():,} rows verified.")

StatementMeta(, db04bf04-00d2-4222-8cf9-9501d4709410, 33, Finished, Available, Finished, False)

INFO:gold.dim_employee.dq:gold.hr.dim_employee written — 290 rows verified.


In [6]:
# ── Read newly added Bronze tables ───────────────────────────────────
df_bea = spark.read.table("lh_adventureworks_bronze.dbo.Person_BusinessEntityAddress")
df_addresstype = spark.read.table("lh_adventureworks_bronze.dbo.Person_AddressType")

logger_cust = get_logger("gold.dim_customer.dq")
logger_cust.info(f"BusinessEntityAddress rows: {df_bea.count()}")
logger_cust.info(f"AddressType rows: {df_addresstype.count()}")

df_bea.printSchema()
df_addresstype.printSchema()

StatementMeta(, b95628a9-9695-46fa-972c-a1fd6e67106c, 14, Finished, Available, Finished, False)

INFO:gold.dim_customer.dq:BusinessEntityAddress rows: 19614


root
 |-- BusinessEntityID: integer (nullable = true)
 |-- AddressID: integer (nullable = true)
 |-- AddressTypeID: integer (nullable = true)
 |-- rowguid: string (nullable = true)
 |-- ModifiedDate: timestamp (nullable = true)

root
 |-- AddressTypeID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- rowguid: string (nullable = true)
 |-- ModifiedDate: timestamp (nullable = true)



In [7]:
# ── Read remaining sources needed for dim_customer ────────────────────
df_sales_orders = spark.read.table("lh_adventureworks_silver.sales.sales_orders")
df_address_silver = spark.read.table("lh_adventureworks_silver.person.address")

# ── Step 1: Get distinct customers from sales_orders ──────────────────
df_customers = df_sales_orders.select(
    "CustomerID", "CustomerFullName", "PersonID", "StoreID"
).distinct()

logger_cust = get_logger("gold.dim_customer.dq")

logger_cust.info(f"Distinct customers: {df_customers.count()}")

# ── Step 2: Resolve AddressType name ──────────────────────────────────
df_addresstype_clean = df_addresstype.select(
    F.col("AddressTypeID").alias("at_AddressTypeID"),
    F.col("Name").alias("AddressTypeName")
)

df_bea_resolved = df_bea.join(
    df_addresstype_clean,
    df_bea["AddressTypeID"] == df_addresstype_clean["at_AddressTypeID"],
    how="left"
).drop("at_AddressTypeID")

# ── Step 3: Pick ONE address per customer — prefer "Main Office" or "Billing", else any ──
# A customer (BusinessEntityID = PersonID or StoreID) may have multiple address types.
# We rank and keep the most relevant one per BusinessEntityID.
from pyspark.sql.window import Window

address_priority = F.when(F.col("AddressTypeName") == "Main Office", 1) \
    .when(F.col("AddressTypeName") == "Billing", 2) \
    .when(F.col("AddressTypeName") == "Home", 3) \
    .otherwise(4)

df_bea_ranked = df_bea_resolved.withColumn("priority", address_priority)

window_spec = Window.partitionBy("BusinessEntityID").orderBy("priority")
df_bea_ranked = df_bea_ranked.withColumn("rank", F.row_number().over(window_spec))
df_bea_primary = df_bea_ranked.filter(F.col("rank") == 1).drop("priority", "rank")

logger_cust.info(f"Primary address records (one per BusinessEntityID): {df_bea_primary.count()}")

# ── Step 4: Join customer -> business entity address -> address details ──
df_bea_primary_clean = df_bea_primary.select(
    F.col("BusinessEntityID").alias("bea_BusinessEntityID"),
    F.col("AddressID").alias("bea_AddressID"),
    "AddressTypeName"
)

df_address_clean = df_address_silver.select(
    F.col("AddressID").alias("addr_AddressID"),
    "AddressLine1", "City", "StateProvinceName",
    "CountryRegionCode", "PostalCode", "TerritoryID"
)

# Customers can be linked via PersonID (individual) OR StoreID (store account)
# Coalesce to get whichever BusinessEntityID applies
df_customers_with_entity = df_customers.withColumn(
    "ResolvedBusinessEntityID",
    F.coalesce(F.col("PersonID"), F.col("StoreID"))
)

df_dim_customer = df_customers_with_entity.join(
    df_bea_primary_clean,
    df_customers_with_entity["ResolvedBusinessEntityID"] == df_bea_primary_clean["bea_BusinessEntityID"],
    how="left"
).drop("bea_BusinessEntityID")

df_dim_customer = df_dim_customer.join(
    df_address_clean,
    df_dim_customer["bea_AddressID"] == df_address_clean["addr_AddressID"],
    how="left"
).drop("bea_AddressID", "addr_AddressID")

actual_rows = df_dim_customer.count()
logger_cust.info(f"dim_customer row count: {actual_rows}")
logger_cust.info(f"{df_dim_customer.printSchema()}")

StatementMeta(, b95628a9-9695-46fa-972c-a1fd6e67106c, 15, Finished, Available, Finished, False)

INFO:gold.dim_customer.dq:Distinct customers: 19119
INFO:gold.dim_customer.dq:Primary address records (one per BusinessEntityID): 19579


root
 |-- CustomerID: integer (nullable = true)
 |-- CustomerFullName: string (nullable = true)
 |-- PersonID: integer (nullable = true)
 |-- StoreID: integer (nullable = true)
 |-- ResolvedBusinessEntityID: integer (nullable = true)
 |-- AddressTypeName: string (nullable = true)
 |-- AddressLine1: string (nullable = true)
 |-- City: string (nullable = true)
 |-- StateProvinceName: string (nullable = true)
 |-- CountryRegionCode: string (nullable = true)
 |-- PostalCode: string (nullable = true)
 |-- TerritoryID: integer (nullable = true)



In [8]:
checks = [
    check_row_count(df_dim_customer, df_customers.count()),
    check_null_pk(df_dim_customer, "CustomerID"),
    check_duplicate_pk(df_dim_customer, "CustomerID"),
]
run_dq_checks(checks, logger_cust, "dim_customer")

# Informational — not a hard fail, just visibility
unmatched_address = df_dim_customer.filter(F.col("AddressLine1").isNull()).count()
total = df_dim_customer.count()
logger_cust.info(
    f"[INFO] Customers with no resolved address: {unmatched_address:,} "
    f"({(unmatched_address/total)*100:.1f}% of {total:,})"
)

StatementMeta(, b95628a9-9695-46fa-972c-a1fd6e67106c, 16, Finished, Available, Finished, False)

INFO:gold.dim_customer.dq:[DQ] [PASS] Row count — Expected 19,119, got 19,119
INFO:gold.dim_customer.dq:[DQ] [PASS] Null PK — CustomerID — 0 nulls
INFO:gold.dim_customer.dq:[DQ] [PASS] Duplicate PK — CustomerID — 0 duplicates
INFO:gold.dim_customer.dq:[DQ] All checks passed for dim_customer.
INFO:gold.dim_customer.dq:[INFO] Customers with no resolved address: 635 (3.3% of 19,119)


In [9]:
# ── Write ──────────────────────────────────────────────────────────────
df_dim_customer.write \
    .format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("lh_adventureworks_gold.sales.dim_customer")

logger_cust.info(f"gold.sales.dim_customer written — {spark.read.table('lh_adventureworks_gold.sales.dim_customer').count():,} rows verified.")

StatementMeta(, b95628a9-9695-46fa-972c-a1fd6e67106c, 17, Finished, Available, Finished, False)

INFO:gold.dim_customer.dq:gold.sales.dim_customer written — 19,119 rows verified.


In [4]:
logger_utm = get_logger("gold.user_territory_mapping.dq")

# ── Read dim_territory to validate against real TerritoryIDs ──────────
df_territory_check = spark.read.table("lh_adventureworks_gold.sales.dim_territory")
df_territory_check.select("TerritoryID", "TerritoryName").show()

StatementMeta(, 20f59592-565c-49db-bfbb-9fb03f52a7ce, 17, Finished, Available, Finished, False)

+-----------+--------------+
|TerritoryID| TerritoryName|
+-----------+--------------+
|          1|     Northwest|
|          2|     Northeast|
|          3|       Central|
|          4|     Southwest|
|          5|     Southeast|
|          6|        Canada|
|          7|        France|
|          8|       Germany|
|          9|     Australia|
|         10|United Kingdom|
+-----------+--------------+



In [5]:
from pyspark.sql import functions as F

# ── Build UserTerritoryMapping ─────────────────────────────────────────
# Test mappings — each fictional Analyst UPN assigned to one territory
mapping_data = [
    ("analyst.northwest@lanacloud.com", 1),
    ("analyst.northeast@lanacloud.com", 2),
    ("analyst.central@lanacloud.com", 3),
    ("analyst.canada@lanacloud.com", 6),
    ("guyfomen@lanacloud.com", 1),  # your own account, for live testing without "View as Roles"
]

df_user_territory = spark.createDataFrame(
    mapping_data,
    schema=["UserPrincipalName", "TerritoryID"]
)

actual_rows = df_user_territory.count()
logger_utm.info(f"UserTerritoryMapping row count: {actual_rows}")
df_user_territory.show(truncate=False)

StatementMeta(, 20f59592-565c-49db-bfbb-9fb03f52a7ce, 18, Finished, Available, Finished, False)

INFO:trident_token_library_wrapper:Calling getAccessToken from PyTridentTokenLibrary


+-------------------------------+-----------+
|UserPrincipalName              |TerritoryID|
+-------------------------------+-----------+
|analyst.northwest@lanacloud.com|1          |
|analyst.northeast@lanacloud.com|2          |
|analyst.central@lanacloud.com  |3          |
|analyst.canada@lanacloud.com   |6          |
|guyfomen@lanacloud.com         |1          |
+-------------------------------+-----------+



In [6]:
# ── DQ checks ─────────────────────────────────────────────────────────
checks = [
    check_row_count(df_user_territory, 5),
    check_null_pk(df_user_territory, "UserPrincipalName"),
    check_duplicate_pk(df_user_territory, "UserPrincipalName"),
    check_referential_integrity(df_user_territory, df_territory_check, "TerritoryID", dim_name="dim_territory"),
]
run_dq_checks(checks, logger_utm, "user_territory_mapping")

# ── Write ──────────────────────────────────────────────────────────────
df_user_territory.write \
    .format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("lh_adventureworks_gold.sales.user_territory_mapping")

logger_utm.info(
    f"gold.sales.user_territory_mapping written — "
    f"{spark.read.table('lh_adventureworks_gold.sales.user_territory_mapping').count():,} rows verified."
)

StatementMeta(, 20f59592-565c-49db-bfbb-9fb03f52a7ce, 19, Finished, Available, Finished, False)

INFO:gold.user_territory_mapping.dq:[DQ] [PASS] Row count — Expected 5, got 5
INFO:gold.user_territory_mapping.dq:[DQ] [PASS] Null PK — UserPrincipalName — 0 nulls
INFO:gold.user_territory_mapping.dq:[DQ] [PASS] Duplicate PK — UserPrincipalName — 0 duplicates
INFO:gold.user_territory_mapping.dq:[DQ] [PASS] Referential integrity — TerritoryID → dim_territory — 0 rows with TerritoryID not found in dim_territory
INFO:gold.user_territory_mapping.dq:[DQ] All checks passed for user_territory_mapping.
INFO:gold.user_territory_mapping.dq:gold.sales.user_territory_mapping written — 5 rows verified.
